# Two-Dimensional Stromatolite Cross-Section Model

This notebook implements a two-dimensional cross-section model by representing the stromatolite surface as a horizontal array of one-dimensional active microbial mats. Each column retains the interpretable biology of the vertical-column model, while local water depth, light, sediment retention, burial, and lateral smoothing vary across space.

The model is deliberately height-field based rather than voxel based. Each timestep records a raster row of newly added material across the transect, so the final output can be visualised as a laminated stromatolite cross-section.

## Model Rules

- The surface is a 1-D horizontal transect of active columns, each with its own height, biomass, EPS, trapped sediment, and local lamina identifier.
- Incident light and temperature follow the same seasonal forcing functions as the 1-D model.
- Water depth is computed locally from the water level and each column height, so elevated parts receive more light.
- Photosynthetic biomass growth follows a Monod-style light response multiplied by temperature suitability.
- Biomass produces EPS, and EPS controls the fraction of supplied sediment retained by each column.
- Sediment supply has shared seasonal and storm forcing plus local spatial weather noise.
- Burial is triggered locally when either one timestep traps enough sediment or cumulative active sediment exceeds the burial threshold.
- Buried columns are reseeded with a new active mat while neighbouring columns may remain in the previous lamina.
- A weak lateral smoothing step limits single-cell spikes and mimics small-scale surface redistribution.

In [ ]:
%run common.ipynb

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, fields
from pathlib import Path
import json
import math

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
IMAGE_DIR = PROJECT_ROOT / "diagrams"
BIOLOGY_CONFIG_FILE = DATA_DIR / "biology.config"
GEOMETRY_CONFIG_FILE = DATA_DIR / "2d.config"

YLORBR_CMAP = plt.get_cmap("YlOrBr")
YLORBR_PALETTE = {
    "pale": YLORBR_CMAP(0.18),
    "light": YLORBR_CMAP(0.32),
    "mid": YLORBR_CMAP(0.50),
    "strong": YLORBR_CMAP(0.68),
    "dark": YLORBR_CMAP(0.86),
    "deep": YLORBR_CMAP(0.98),
}


In [ ]:
@dataclass
class ModelParameters2D(ModelParameters1D):
    """Container for the 2-D cross-section model parameters.
    """
    nx: int
    dx_mm: float
    initial_surface_noise_mm: float
    lateral_smoothing_rate: float
    spatial_sediment_noise: float
    snapshot_count: int

In [ ]:
def load_parameter_values(parameter_path: Path) -> dict:
    """Load named parameter values from a JSON config file.

    :param parameter_path: Path to a JSON config file with a parameters object.
    :return: Mapping from parameter names to raw values.
    """
    with parameter_path.open() as handle:
        parameter_document = json.load(handle)

    return {
        name: parameter["value"]
        for name, parameter in parameter_document["parameters"].items()
    }


def load_2d_parameters(biology_path: Path, geometry_path: Path) -> ModelParameters2D:
    """Load shared biology and 2-D geometry configuration.

    :param biology_path: Path to the shared biology JSON config file.
    :param geometry_path: Path to the 2-D geometry JSON config file.
    :return: A populated 2-D parameters instance.
    """
    raw_values = load_parameter_values(biology_path)
    raw_values.update(load_parameter_values(geometry_path))

    integer_parameters = {"time_steps", "random_seed", "nx", "snapshot_count"}
    for name in integer_parameters & raw_values.keys():
        raw_values[name] = int(raw_values[name])

    allowed_names = {field.name for field in fields(ModelParameters2D)}
    filtered_values = {
        name: value
        for name, value in raw_values.items()
        if name in allowed_names
    }
    return ModelParameters2D(**filtered_values)


In [ ]:
def seasonal_light(day: float, params: ModelParameters2D) -> float:
    """Compute seasonally forced incident light for a day of year.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling mean light, amplitude, peak day, and minimum light.
    :return: Relative incident light after seasonal forcing and lower clamping.
    """
    phase = 2.0 * math.pi * (day - params.light_peak_day) / params.days_per_year
    forced_light = params.surface_light * (1.0 + params.light_amplitude * math.cos(phase))
    return max(params.min_surface_light, forced_light)

In [ ]:
def seasonal_temperature(day: float, params: ModelParameters2D) -> float:
    """Compute seasonally forced temperature for a day of year.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling mean temperature, amplitude, and peak day.
    :return: Temperature in degrees Celsius.
    """
    phase = 2.0 * math.pi * (day - params.temperature_peak_day) / params.days_per_year
    return params.mean_temperature_c + params.temperature_amplitude_c * math.cos(phase)

In [ ]:
def day_of_year(step: int, dt_days: float, days_per_year: float) -> float:
    """Convert a timestep into a repeating day-of-year value.

    :param step: Current simulation step.
    :param dt_days: Duration of each timestep in days.
    :param days_per_year: Number of days in one seasonal cycle.
    :return: Day within the seasonal cycle.
    """
    return (step * dt_days) % days_per_year

In [ ]:
def temperature_response(temperature_c: float, optimal_temperature_c: float, temperature_width_c: float) -> float:
    """Compute biological suitability from temperature.

    :param temperature_c: Current temperature in degrees Celsius.
    :param optimal_temperature_c: Temperature where the multiplier reaches one.
    :param temperature_width_c: Width of the bell-shaped response curve.
    :return: Biological rate multiplier between zero and one.
    """
    scaled_difference = (temperature_c - optimal_temperature_c) / temperature_width_c
    return math.exp(-(scaled_difference ** 2))

In [ ]:
def light_limitation(light: np.ndarray, half_saturation_light: float) -> np.ndarray:
    """Compute the Monod-style light limitation factor.

    :param light: Local relative light intensity.
    :param half_saturation_light: Light intensity where growth is half-maximal.
    :return: Photosynthetic light limitation factor between zero and one.
    """
    return light / (half_saturation_light + light)

In [ ]:
def monod_growth_rate(light: np.ndarray, max_growth_rate: float, half_saturation_light: float) -> np.ndarray:
    """Compute light-limited photosynthetic growth rates.

    :param light: Local relative light intensity at each column.
    :param max_growth_rate: Maximum possible growth rate per day.
    :param half_saturation_light: Light intensity where growth is half-maximal.
    :return: Growth rates per day for all columns.
    """
    return max_growth_rate * light_limitation(light, half_saturation_light)

In [ ]:
def trapping_fraction(eps: np.ndarray, max_efficiency: float, eps_half_saturation: float) -> np.ndarray:
    """Compute the fraction of supplied sediment trapped by EPS.

    :param eps: Relative EPS concentration in each active mat.
    :param max_efficiency: Maximum possible trapping fraction.
    :param eps_half_saturation: EPS level where trapping reaches half its maximum.
    :return: Fraction of incoming sediment retained by each column.
    """
    return max_efficiency * eps / (eps_half_saturation + eps)

In [ ]:
def water_depth_above_surface(height_mm: np.ndarray, params: ModelParameters2D) -> np.ndarray:
    """Compute local water depth above each active mat surface.

    :param height_mm: Current stromatolite height at each x-position.
    :param params: Model parameters controlling water level and minimum water cover.
    :return: Water depth above each active mat in millimetres.
    """
    return np.maximum(params.minimum_water_cover_mm, params.water_depth_mm - height_mm)

In [ ]:
def light_at_surface(day: float, height_mm: np.ndarray, params: ModelParameters2D) -> dict:
    """Compute light reaching each active mat through local overlying water.

    :param day: Day within the seasonal cycle.
    :param height_mm: Current stromatolite height at each x-position.
    :param params: Model parameters controlling incident light and water attenuation.
    :return: Dictionary containing incident light, local water depth, and local mat light.
    """
    incident_light = seasonal_light(day, params)
    water_depth_mm = water_depth_above_surface(height_mm, params)
    mat_light = incident_light * np.exp(-params.water_light_attenuation_per_mm * water_depth_mm)
    return {
        "incident_light": incident_light,
        "water_depth_above_mat_mm": water_depth_mm,
        "light_at_mat": mat_light,
    }

In [ ]:
def seasonal_sediment_multiplier(day: float, params: ModelParameters2D) -> float:
    """Compute the slow seasonal multiplier for sediment supply.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling sediment seasonality and lower clamping.
    :return: Multiplicative sediment-supply factor.
    """
    phase = 2.0 * math.pi * (day - params.sediment_peak_day) / params.days_per_year
    multiplier = 1.0 + params.sediment_seasonal_amplitude * math.cos(phase)
    return max(params.min_sediment_multiplier, multiplier)

In [ ]:
def daily_sediment_weather_multiplier(noise_fraction: float, rng: np.random.Generator) -> float:
    """Draw a non-negative transect-wide weather multiplier for sediment supply.

    :param noise_fraction: Standard deviation of daily weather variation as a fraction of the mean.
    :param rng: NumPy random generator used for reproducible draws.
    :return: Daily multiplicative sediment-supply factor.
    """
    return max(0.0, rng.normal(loc=1.0, scale=noise_fraction))

In [ ]:
def storm_sediment_pulse(params: ModelParameters2D, rng: np.random.Generator) -> tuple[float, bool]:
    """Draw a rare storm sediment pulse for one timestep.

    :param params: Model parameters controlling storm probability and pulse size.
    :param rng: NumPy random generator used for reproducible storm timing and size.
    :return: Additional sediment supply in millimetres per day and whether a storm occurred.
    """
    storm_probability = min(1.0, params.storm_probability_per_day * params.dt_days)
    storm_today = rng.random() < storm_probability
    if not storm_today:
        return 0.0, False

    pulse = rng.normal(
        loc=params.storm_sediment_mm_per_day,
        scale=params.storm_sediment_mm_per_day * params.storm_noise_fraction,
    )
    return max(0.0, pulse), True

In [ ]:
def spatial_sediment_supply(day: float, params: ModelParameters2D, rng: np.random.Generator) -> dict:
    """Combine shared forcing with spatially variable sediment supply.

    :param day: Day within the seasonal cycle.
    :param params: Model parameters controlling all sediment-supply forcing terms.
    :param rng: NumPy random generator used for reproducible stochastic forcing.
    :return: Dictionary containing sediment supply components in millimetres per day.
    """
    seasonal_multiplier = seasonal_sediment_multiplier(day, params)
    weather_multiplier = daily_sediment_weather_multiplier(params.sediment_noise_fraction, rng)
    storm_pulse, storm_today = storm_sediment_pulse(params, rng)
    background_supply = params.sediment_input_mm_per_day * seasonal_multiplier * weather_multiplier
    total_supply = background_supply + storm_pulse
    spatial_multiplier = np.maximum(0.0, rng.normal(loc=1.0, scale=params.spatial_sediment_noise, size=params.nx))

    return {
        "sediment_baseline_mm_per_day": params.sediment_input_mm_per_day,
        "sediment_seasonal_multiplier": seasonal_multiplier,
        "sediment_weather_multiplier": weather_multiplier,
        "storm_sediment_mm_per_day": storm_pulse,
        "storm_today": storm_today,
        "sediment_supply_mm_per_day": total_supply,
        "local_sediment_supply_mm_per_day": total_supply * spatial_multiplier,
    }

In [ ]:
def initialise_2d_state(params: ModelParameters2D, rng: np.random.Generator) -> dict:
    """Create the initial active surface state for the cross-section.

    :param params: Model parameters controlling transect size and starting relief.
    :param rng: NumPy random generator used for reproducible surface roughness.
    :return: Mutable model state arrays.
    """
    x_mm = np.arange(params.nx, dtype=float) * params.dx_mm
    centre = 0.5 * (params.nx - 1) * params.dx_mm
    width = max(params.dx_mm, 0.22 * params.nx * params.dx_mm)
    starter_dome = 0.25 * params.initial_surface_noise_mm * np.exp(-((x_mm - centre) ** 2) / (2.0 * width ** 2))
    surface_noise = rng.normal(loc=0.0, scale=params.initial_surface_noise_mm, size=params.nx)
    height = np.maximum(0.05, params.initial_mat_thickness_mm + starter_dome + surface_noise)

    return {
        "x_mm": x_mm,
        "height_mm": height,
        "biomass": np.full(params.nx, params.new_layer_biomass_seed, dtype=float),
        "eps": np.full(params.nx, params.new_layer_eps_seed, dtype=float),
        "active_sediment_mm": np.zeros(params.nx, dtype=float),
        "layer_id": np.zeros(params.nx, dtype=int),
    }

In [ ]:
def lateral_laplacian(values: np.ndarray) -> np.ndarray:
    """Compute a one-dimensional nearest-neighbour Laplacian.

    :param values: Surface values along the x transect.
    :return: Difference between local values and neighbouring average.
    """
    padded = np.pad(values, pad_width=1, mode="edge")
    return padded[:-2] - 2.0 * padded[1:-1] + padded[2:]

In [ ]:
def apply_lateral_smoothing(height_mm: np.ndarray, params: ModelParameters2D) -> np.ndarray:
    """Apply weak lateral surface smoothing.

    :param height_mm: Current stromatolite height at each x-position.
    :param params: Model parameters controlling smoothing strength.
    :return: Smoothed height field.
    """
    smoothing = params.lateral_smoothing_rate * lateral_laplacian(height_mm)
    return np.maximum(0.0, height_mm + smoothing)

In [ ]:
def update_active_surface(state: dict, params: ModelParameters2D, step: int, rng: np.random.Generator) -> dict:
    """Advance all exposed microbial mats by one timestep.

    :param state: Mutable 2-D model state arrays.
    :param params: Model parameters controlling growth, sediment trapping, and burial.
    :param step: Current simulation step.
    :param rng: NumPy random generator used for sediment-supply variation.
    :return: Dictionary of timestep diagnostics and raster rows.
    """
    # Place this timestep within the annual cycle so light, temperature, and sediment
    # forcing represent the current season experienced by the microbial mat.
    current_day = day_of_year(step, params.dt_days, params.days_per_year)
    light_forcing = light_at_surface(current_day, state["height_mm"], params)
    temperature_today = seasonal_temperature(current_day, params)
    temp_factor = temperature_response(
        temperature_today,
        params.optimal_temperature_c,
        params.temperature_width_c,
    )
    local_light = light_forcing["light_at_mat"]

    # Convert local mat light and temperature suitability into biological production.
    # Higher columns have shallower water cover, so they can receive more light and
    # grow differently from lower neighbouring columns.
    growth_rate = monod_growth_rate(local_light, params.max_growth_rate_per_day, params.half_saturation_light) * temp_factor
    eps_production_rate = params.eps_production_rate_per_day * temp_factor
    biomass_change = (growth_rate - params.mortality_rate_per_day) * state["biomass"] * params.dt_days
    eps_change = (eps_production_rate * state["biomass"] - params.eps_decay_rate_per_day * state["eps"]) * params.dt_days

    # Deliver sediment from the water column, then let EPS decide how much is retained
    # on each part of the surface rather than passing over the mat.
    sediment_forcing = spatial_sediment_supply(current_day, params, rng)
    supplied_sediment = sediment_forcing["local_sediment_supply_mm_per_day"] * params.dt_days
    retained_fraction = trapping_fraction(state["eps"], params.trapping_efficiency, params.eps_half_saturation)
    trapped_sediment = supplied_sediment * retained_fraction

    # Translate newly produced biomass, EPS, and trapped sediment into vertical accretion
    # of the stromatolite surface for this day.
    positive_biomass_change = np.maximum(0.0, biomass_change)
    positive_eps_change = np.maximum(0.0, eps_change)
    biological_increment = (
        positive_biomass_change * params.biomass_to_thickness
        + positive_eps_change * params.eps_to_thickness
    )
    sediment_increment = trapped_sediment * params.sediment_to_thickness
    total_increment = biological_increment + sediment_increment

    state["biomass"] = np.maximum(0.0, state["biomass"] + biomass_change)
    state["eps"] = np.maximum(0.0, state["eps"] + eps_change)
    state["active_sediment_mm"] += trapped_sediment
    state["height_mm"] += total_increment

    # Identify columns where sediment has locally covered the active mat. Those columns
    # close one lamina and start a fresh microbial surface, while unburied neighbours
    # continue growing in their existing lamina.
    buried = (
        (trapped_sediment >= params.rapid_burial_sediment_threshold_mm)
        | (state["active_sediment_mm"] >= params.burial_sediment_threshold_mm)
    )
    if buried.any():
        state["layer_id"][buried] += 1
        state["active_sediment_mm"][buried] = 0.0
        state["biomass"][buried] = params.new_layer_biomass_seed
        state["eps"][buried] = params.new_layer_eps_seed
        state["height_mm"][buried] += params.initial_mat_thickness_mm
        total_increment = total_increment.copy()
        biological_increment = biological_increment.copy()
        total_increment[buried] += params.initial_mat_thickness_mm
        biological_increment[buried] += params.initial_mat_thickness_mm

    # Let neighbouring columns share a little relief after growth and burial, representing
    # small-scale slumping, erosion, or lateral mat continuity without erasing the surface.
    state["height_mm"] = apply_lateral_smoothing(state["height_mm"], params)

    # Record the composition of the material deposited during this timestep, as opposed
    # to the current active surface state.
    sediment_fraction = np.divide(
        sediment_increment,
        total_increment,
        out=np.zeros_like(total_increment),
        where=total_increment > 0,
    )

    biological_signal = np.divide(
        biological_increment,
        total_increment,
        out=np.zeros_like(total_increment),
        where=total_increment > 0,
    )

    return {
        "step": step + 1,
        "day_of_year": current_day,
        "incident_light": light_forcing["incident_light"],
        "temperature_c": temperature_today,
        "temperature_factor": temp_factor,
        "sediment_seasonal_multiplier": sediment_forcing["sediment_seasonal_multiplier"],
        "sediment_weather_multiplier": sediment_forcing["sediment_weather_multiplier"],
        "storm_sediment_mm_per_day": sediment_forcing["storm_sediment_mm_per_day"],
        "storm_today": sediment_forcing["storm_today"],
        "sediment_supply_mm_per_day": sediment_forcing["sediment_supply_mm_per_day"],
        "mean_height_mm": float(np.mean(state["height_mm"])),
        "max_height_mm": float(np.max(state["height_mm"])),
        "relief_mm": float(np.max(state["height_mm"]) - np.min(state["height_mm"])),
        "mean_biomass": float(np.mean(state["biomass"])),
        "mean_eps": float(np.mean(state["eps"])),
        "mean_active_sediment_mm": float(np.mean(state["active_sediment_mm"])),
        "mean_light_at_mat": float(np.mean(local_light)),
        "min_light_at_mat": float(np.min(local_light)),
        "max_light_at_mat": float(np.max(local_light)),
        "mean_water_depth_above_mat_mm": float(np.mean(light_forcing["water_depth_above_mat_mm"])),
        "mean_supplied_sediment_mm": float(np.mean(supplied_sediment)),
        "mean_trapped_sediment_mm": float(np.mean(trapped_sediment)),
        "burial_count": int(np.count_nonzero(buried)),
        "surface_height_mm": state["height_mm"].copy(),
        "biomass_profile": state["biomass"].copy(),
        "eps_profile": state["eps"].copy(),
        "active_sediment_profile_mm": state["active_sediment_mm"].copy(),
        "light_profile": local_light.copy(),
        "deposition_increment_mm": total_increment.copy(),
        "sediment_fraction": sediment_fraction.copy(),
        "biological_signal": biological_signal.copy(),
        "buried": buried.copy(),
        "layer_id": state["layer_id"].copy(),
    }

In [ ]:
def summarise_initial_state(state: dict) -> dict:
    """Summarise the initial cross-section before any timestep update.

    :param state: Mutable 2-D model state arrays.
    :return: Summary metrics for the initial state.
    """
    return {
        "step": 0,
        "day_of_year": 0.0,
        "incident_light": np.nan,
        "temperature_c": np.nan,
        "temperature_factor": np.nan,
        "sediment_seasonal_multiplier": np.nan,
        "sediment_weather_multiplier": np.nan,
        "storm_sediment_mm_per_day": 0.0,
        "storm_today": False,
        "sediment_supply_mm_per_day": np.nan,
        "mean_height_mm": float(np.mean(state["height_mm"])),
        "max_height_mm": float(np.max(state["height_mm"])),
        "relief_mm": float(np.max(state["height_mm"]) - np.min(state["height_mm"])),
        "mean_biomass": float(np.mean(state["biomass"])),
        "mean_eps": float(np.mean(state["eps"])),
        "mean_active_sediment_mm": float(np.mean(state["active_sediment_mm"])),
        "mean_light_at_mat": np.nan,
        "min_light_at_mat": np.nan,
        "max_light_at_mat": np.nan,
        "mean_water_depth_above_mat_mm": np.nan,
        "mean_supplied_sediment_mm": 0.0,
        "mean_trapped_sediment_mm": 0.0,
        "burial_count": 0,
        "surface_height_mm": state["height_mm"].copy(),
        "biomass_profile": state["biomass"].copy(),
        "eps_profile": state["eps"].copy(),
        "active_sediment_profile_mm": state["active_sediment_mm"].copy(),
        "light_profile": np.full_like(state["height_mm"], np.nan),
    }

In [ ]:
def run_2d_model(params: ModelParameters2D) -> tuple[dict, list[dict], dict, list[dict]]:
    """Run the 2-D stromatolite cross-section model.

    :param params: Model parameters controlling growth, sediment trapping, burial, and spatial coupling.
    :return: Final state, timestep history, stratigraphic rasters, and selected surface snapshots.
    """
    # Seed one reproducible cross-section and initialise the exposed microbial surface
    # before any seasonal forcing, growth, or burial has occurred.
    rng = np.random.default_rng(params.random_seed)
    state = initialise_2d_state(params, rng)
    history = [summarise_initial_state(state)]

    # Select a small set of times that will show how the stromatolite surface changes
    # from the initial mat into the final mounded cross-section.
    snapshot_steps = set(np.linspace(0, params.time_steps, params.snapshot_count, dtype=int))
    snapshots = []
    if 0 in snapshot_steps:
        snapshots.append({
            "step": 0,
            "surface_height_mm": state["height_mm"].copy(),
            "biomass_profile": state["biomass"].copy(),
            "eps_profile": state["eps"].copy(),
            "active_sediment_profile_mm": state["active_sediment_mm"].copy(),
            "light_profile": np.full(params.nx, np.nan),
        })

    # These rows become the 2-D stratigraphic archive: each timestep contributes one
    # laterally varying depositional layer across the transect.
    deposition_rows = []
    sediment_fraction_rows = []
    biological_signal_rows = []
    burial_rows = []
    layer_id_rows = []

    for step in range(params.time_steps):
        # Advance the active surface through one day of light exposure, biological
        # growth, sediment trapping, local burial, and weak lateral smoothing.
        diagnostics = update_active_surface(state, params, step, rng)
        history.append(diagnostics)

        # Store the new material and burial state as one horizontal slice of the final
        # laminated cross-section.
        deposition_rows.append(diagnostics["deposition_increment_mm"])
        sediment_fraction_rows.append(diagnostics["sediment_fraction"])
        biological_signal_rows.append(diagnostics["biological_signal"])
        burial_rows.append(diagnostics["buried"].astype(float))
        layer_id_rows.append(diagnostics["layer_id"].astype(float))

        # Preserve selected exposed-surface states so the model can be inspected as a
        # growth sequence, not only as a finished stratigraphic section.
        if diagnostics["step"] in snapshot_steps:
            snapshots.append({
                "step": diagnostics["step"],
                "surface_height_mm": diagnostics["surface_height_mm"].copy(),
                "biomass_profile": diagnostics["biomass_profile"].copy(),
                "eps_profile": diagnostics["eps_profile"].copy(),
                "active_sediment_profile_mm": diagnostics["active_sediment_profile_mm"].copy(),
                "light_profile": diagnostics["light_profile"].copy(),
            })

    # Stack the daily depositional slices into raster grids for plotting laminae,
    # sediment-rich intervals, biological signal, and local burial through time.
    stratigraphy = {
        "deposition_increment_mm": np.vstack(deposition_rows),
        "sediment_fraction": np.vstack(sediment_fraction_rows),
        "biological_signal": np.vstack(biological_signal_rows),
        "burial_map": np.vstack(burial_rows),
        "layer_id": np.vstack(layer_id_rows),
    }
    return state, history, stratigraphy, snapshots

In [ ]:
def history_as_array(history: list[dict], field: str) -> np.ndarray:
    """Extract a named scalar history field as a NumPy array.

    :param history: List of timestep summary dictionaries.
    :param field: Dictionary key to extract from each summary.
    :return: Numeric array containing the requested field over time.
    """
    return np.array([row.get(field, np.nan) for row in history], dtype=float)

In [ ]:
def stratigraphy_image(stratigraphy: dict, field: str) -> np.ndarray:
    """Convert a timestep raster into a bottom-to-top plotting image.

    :param stratigraphy: Dictionary of stratigraphic raster arrays.
    :param field: Raster name to extract.
    :return: Two-dimensional image array with earliest deposition at the bottom.
    """
    return np.flipud(stratigraphy[field])

In [ ]:
def plot_growth_summary_2d(history: list[dict], params: ModelParameters2D) -> None:
    """Plot mean height, relief, burial counts, and material summaries through time.

    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters used to convert steps into days.
    :return: None. A Matplotlib figure is displayed.
    """
    time_days = history_as_array(history, "step") * params.dt_days
    _, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)

    axes[0].plot(time_days, history_as_array(history, "mean_height_mm"), color=YLORBR_PALETTE["strong"])
    axes[0].plot(time_days, history_as_array(history, "max_height_mm"), color=YLORBR_PALETTE["dark"], linewidth=1.0, alpha=0.8)
    axes[0].set_title("Surface height")
    axes[0].set_xlabel("Time (days)")
    axes[0].set_ylabel("Height (mm)")

    axes[1].plot(time_days, history_as_array(history, "relief_mm"), color=YLORBR_PALETTE["deep"])
    axes[1].set_title("Surface relief")
    axes[1].set_xlabel("Time (days)")
    axes[1].set_ylabel("Relief (mm)")

    axes[2].plot(time_days, history_as_array(history, "burial_count"), color=YLORBR_PALETTE["deep"], linewidth=1.0)
    axes[2].set_title("Local burial events")
    axes[2].set_xlabel("Time (days)")
    axes[2].set_ylabel("Columns buried")

    axes[3].plot(time_days, history_as_array(history, "mean_biomass"), label="biomass", color=YLORBR_PALETTE["light"])
    axes[3].plot(time_days, history_as_array(history, "mean_eps"), label="EPS", color=YLORBR_PALETTE["mid"])
    axes[3].plot(time_days, history_as_array(history, "mean_active_sediment_mm"), label="active sediment", color=YLORBR_PALETTE["dark"])
    axes[3].set_title("Mean active materials")
    axes[3].set_xlabel("Time (days)")
    axes[3].legend(frameon=False)

    plt.savefig(IMAGE_DIR / "2d-growth-summary.png", dpi=300)
    plt.show()

In [ ]:
def plot_environmental_forcing_2d(history: list[dict], params: ModelParameters2D) -> None:
    """Plot environmental forcing and spatial light range through time.

    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters used to convert steps into days.
    :return: None. A Matplotlib figure is displayed.
    """
    time_days = history_as_array(history, "step") * params.dt_days
    _, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)

    axes[0].plot(time_days, history_as_array(history, "incident_light"), label="incident", color=YLORBR_PALETTE["mid"])
    axes[0].plot(time_days, history_as_array(history, "mean_light_at_mat"), label="mean mat", color=YLORBR_PALETTE["dark"])
    axes[0].fill_between(
        time_days,
        history_as_array(history, "min_light_at_mat"),
        history_as_array(history, "max_light_at_mat"),
        color=YLORBR_PALETTE["pale"],
        alpha=0.35,
        linewidth=0.0,
        label="mat range",
    )
    axes[0].set_title("Spatial mat light")
    axes[0].set_xlabel("Time (days)")
    axes[0].set_ylabel("Relative light")
    axes[0].legend(frameon=False)

    axes[1].plot(time_days, history_as_array(history, "mean_water_depth_above_mat_mm"), color=YLORBR_PALETTE["strong"])
    axes[1].set_title("Mean water above mat")
    axes[1].set_xlabel("Time (days)")
    axes[1].set_ylabel("Depth (mm)")

    axes[2].plot(time_days, history_as_array(history, "temperature_c"), color=YLORBR_PALETTE["deep"])
    axes[2].set_title("Seasonal temperature")
    axes[2].set_xlabel("Time (days)")
    axes[2].set_ylabel("Temperature (deg C)")

    axes[3].plot(time_days, history_as_array(history, "sediment_supply_mm_per_day"), color=YLORBR_PALETTE["dark"], linewidth=1.0)
    storm_mask = history_as_array(history, "storm_today") > 0.0
    if storm_mask.any():
        axes[3].scatter(
            time_days[storm_mask],
            history_as_array(history, "sediment_supply_mm_per_day")[storm_mask],
            color=YLORBR_PALETTE["deep"],
            s=18,
            label="storm",
        )
        axes[3].legend(frameon=False)
    axes[3].set_title("Sediment forcing")
    axes[3].set_xlabel("Time (days)")
    axes[3].set_ylabel("mm/day")

    plt.savefig(IMAGE_DIR / "2d-environmental-forcing.png", dpi=300)
    plt.show()

In [ ]:
def plot_surface_snapshots(snapshots: list[dict], params: ModelParameters2D) -> None:
    """Plot selected surface-height profiles through time.

    :param snapshots: List of saved surface snapshots.
    :param params: Model parameters controlling horizontal spacing.
    :return: None. A Matplotlib figure is displayed.
    """
    _, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
    for snapshot in snapshots:
        color_position = snapshot["step"] / max(1, params.time_steps)
        ax.plot(
            np.arange(params.nx) * params.dx_mm,
            snapshot["surface_height_mm"],
            color=YLORBR_CMAP(0.25 + 0.70 * color_position),
            linewidth=1.5,
            label=f"day {snapshot['step'] * params.dt_days:.0f}",
        )
    ax.set_title("Surface height through time")
    ax.set_xlabel("x-position (mm)")
    ax.set_ylabel("Height above base (mm)")
    ax.legend(frameon=False, ncol=2)

    plt.savefig(IMAGE_DIR / "2d-surface-snapshots.png", dpi=300)
    plt.show()

In [ ]:
def plot_height_time_heatmap(history: list[dict], params: ModelParameters2D) -> None:
    """Plot x-position versus time coloured by surface height.

    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters controlling horizontal spacing and time conversion.
    :return: None. A Matplotlib figure is displayed.
    """
    height_rows = np.vstack([row["surface_height_mm"] for row in history])
    extent = [0.0, (params.nx - 1) * params.dx_mm, 0.0, params.time_steps * params.dt_days]
    _, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
    image = ax.imshow(height_rows, aspect="auto", origin="lower", extent=extent, cmap=YLORBR_CMAP)
    ax.set_title("Height-time map")
    ax.set_xlabel("x-position (mm)")
    ax.set_ylabel("Time (days)")
    plt.colorbar(image, ax=ax, label="Surface height (mm)")

    plt.savefig(IMAGE_DIR / "2d-height-time-map.png", dpi=300)
    plt.show()

In [ ]:
def plot_burial_event_map(stratigraphy: dict, params: ModelParameters2D) -> None:
    """Plot local burial events through time and x-position.

    :param stratigraphy: Dictionary of stratigraphic raster arrays.
    :param params: Model parameters controlling horizontal spacing and time conversion.
    :return: None. A Matplotlib figure is displayed.
    """
    extent = [0.0, (params.nx - 1) * params.dx_mm, 0.0, params.time_steps * params.dt_days]
    _, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
    image = ax.imshow(stratigraphy["burial_map"], aspect="auto", origin="lower", extent=extent, cmap=YLORBR_CMAP, vmin=0.0, vmax=1.0)
    ax.set_title("Local burial-event map")
    ax.set_xlabel("x-position (mm)")
    ax.set_ylabel("Time (days)")
    plt.colorbar(image, ax=ax, label="Burial event")

    plt.savefig(IMAGE_DIR / "2d-burial-event-map.png", dpi=300)
    plt.show()

In [ ]:
def plot_final_cross_section(stratigraphy: dict, params: ModelParameters2D, final_state) -> None:
    """Plot the final 2-D laminated cross-section.

    :param stratigraphy: Dictionary of stratigraphic raster arrays.
    :param params: Model parameters controlling horizontal and vertical plotting extents.
    :return: None. A Matplotlib figure is displayed.
    """
    deposition = stratigraphy["deposition_increment_mm"]
    sediment_fraction = stratigraphy["sediment_fraction"]

    cumulative_height = np.cumsum(deposition, axis=0)
    final_height = final_state["height_mm"]

    x = np.arange(params.nx) * params.dx_mm

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.imshow(
        sediment_fraction,
        aspect="auto",
        origin="lower",
        extent=[x.min(), x.max(), 0, cumulative_height.max()],
        cmap=YLORBR_CMAP,
    )

    ax.plot(
        x,
        final_height,
        color=YLORBR_PALETTE["pale"],
        linewidth=2.5,
        label="Final smoothed surface",
    )

    ax.set_xlabel("Distance along transect (mm)")
    ax.set_ylabel("Height above base (mm)")
    ax.set_title("Final 2-D stromatolite cross-section")
    ax.legend()
    fig.savefig(IMAGE_DIR / "2d-final-cross-section.png", dpi=300)

    return fig, ax

In [ ]:
def plot_final_surface_diagnostics(final_state: dict, history: list[dict], params: ModelParameters2D) -> None:
    """Plot final surface height, biomass, EPS, sediment, and light profiles.

    :param final_state: Final mutable state returned by the model.
    :param history: List of timestep summary dictionaries from the model run.
    :param params: Model parameters controlling horizontal spacing.
    :return: None. A Matplotlib figure is displayed.
    """
    x_mm = final_state["x_mm"]
    final_light = history[-1]["light_profile"]
    _, axes = plt.subplots(5, 1, figsize=(11, 11), sharex=True, constrained_layout=True)

    axes[0].plot(x_mm, final_state["height_mm"], color=YLORBR_PALETTE["strong"])
    axes[0].set_title("Final height")
    axes[0].set_ylabel("mm")

    axes[1].plot(x_mm, final_state["biomass"], color=YLORBR_PALETTE["light"])
    axes[1].set_title("Final biomass")
    axes[1].set_ylabel("relative")

    axes[2].plot(x_mm, final_state["eps"], color=YLORBR_PALETTE["mid"])
    axes[2].set_title("Final EPS")
    axes[2].set_ylabel("relative")

    axes[3].plot(x_mm, final_state["active_sediment_mm"], color=YLORBR_PALETTE["dark"])
    axes[3].set_title("Final active sediment")
    axes[3].set_ylabel("mm")

    axes[4].plot(x_mm, final_light, color=YLORBR_PALETTE["strong"])
    axes[4].set_title("Final light at mat")
    axes[4].set_xlabel("x-position (mm)")
    axes[4].set_ylabel("relative")

    plt.savefig(IMAGE_DIR / "2d-final-surface-diagnostics.png", dpi=300)
    plt.show()

In [ ]:
def save_final_surface(final_state: dict, output_path: Path) -> None:
    """Save the final 2-D surface state as a CSV file.

    :param final_state: Final mutable state returned by the model.
    :param output_path: Destination CSV path.
    :return: None. The CSV file is written to disk.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    table = np.column_stack([
        final_state["x_mm"],
        final_state["height_mm"],
        final_state["biomass"],
        final_state["eps"],
        final_state["active_sediment_mm"],
        final_state["layer_id"],
    ])
    header = "x_mm,height_mm,biomass,eps,active_sediment_mm,layer_id"
    np.savetxt(output_path, table, delimiter=",", header=header, comments="", fmt="%.6f")

In [ ]:
def save_stratigraphy_rasters(stratigraphy: dict, output_path: Path) -> None:
    """Save the core stratigraphic raster arrays as a compressed NumPy archive.

    :param stratigraphy: Dictionary of stratigraphic raster arrays.
    :param output_path: Destination NPZ path.
    :return: None. The NPZ file is written to disk.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output_path, **stratigraphy)

## Run The Simulation

In [ ]:
params = load_2d_parameters(BIOLOGY_CONFIG_FILE, GEOMETRY_CONFIG_FILE)
final_state, history, stratigraphy, snapshots = run_2d_model(params)

print(f"Final mean height: {history[-1]['mean_height_mm']:.2f} mm")
print(f"Final maximum height: {history[-1]['max_height_mm']:.2f} mm")
print(f"Final surface relief: {history[-1]['relief_mm']:.2f} mm")
print(f"Total local burial events: {int(stratigraphy['burial_map'].sum())}")


In [ ]:
plot_growth_summary_2d(history, params)

In [ ]:
plot_environmental_forcing_2d(history, params)

In [ ]:
plot_surface_snapshots(snapshots, params)

In [ ]:
plot_height_time_heatmap(history, params)

In [ ]:
plot_burial_event_map(stratigraphy, params)

In [ ]:
plot_final_cross_section(stratigraphy, params, final_state)

In [ ]:
plot_final_surface_diagnostics(final_state, history, params)

## Save The 2-D Output

In [ ]:
surface_output_file = DATA_DIR / "2d-final-surface.csv"
stratigraphy_output_file = DATA_DIR / "2d-stratigraphy.npz"

save_final_surface(final_state, surface_output_file)
save_stratigraphy_rasters(stratigraphy, stratigraphy_output_file)

print(f"Saved final surface to {surface_output_file}")
print(f"Saved stratigraphy rasters to {stratigraphy_output_file}")